# Faster R-CNN + MobileNetV3-Large + FPN — CLCXray baseline

Trained fully from scratch (no pretrained backbone or detection head) on the same `CLCXray_YOLO` dataset — same images, same train/val/test split, same crop + cut-and-paste augmentation — used by the team's other two baseline models, for a fair comparison in the results table.

**Before running**: Notebook settings (right panel) → Accelerator → GPU. Add the shared team dataset as an input first ("Add Input" on the right panel).

## 1. Kaggle setup

In [ ]:
# Find where the dataset actually landed (zip nesting can vary)
!find /kaggle/input -maxdepth 3

In [ ]:
# Copy into the writable working directory
# (replace <your-dataset-slug> with what the cell above showed)
!cp -r /kaggle/input/<your-dataset-slug>/CLCXray_YOLO /kaggle/working/

In [ ]:
# Extra packages this pipeline needs beyond what Kaggle preinstalls
!pip install -q torchmetrics supervision fvcore

## 2. Imports and configuration

In [ ]:
import math
from pathlib import Path

import torch
import torchvision
from PIL import Image
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn

DATASET_ROOT = Path("/kaggle/working/CLCXray_YOLO")
NUM_CLASSES = 13   # 12 CLCXray classes + 1 background (torchvision reserves class 0)
EPOCHS = 40      # fine-tuning a pretrained model converges much faster than
                 # training from scratch did -- no need for 100+ epochs here
PATIENCE = 10
BATCH_SIZE = 8     # Faster R-CNN uses much more memory per image than YOLO;
                   # start here and drop to 4 if you hit an out-of-memory error
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["blade", "scissors", "knife", "dagger", "SwissArmyKnife", "PlasticBottle",
               "Cans", "VacuumCup", "GlassBottle", "CartonDrinks", "Tin", "SprayCans"]
print(DEVICE)

## 3. Dataset

Converts YOLO-format labels (normalized `cx,cy,w,h`) into what torchvision's detection models expect — absolute-pixel `x1,y1,x2,y2`, plus a `+1` label offset since class `0` is reserved for "background".

In [ ]:
class CLCXrayDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.image_files = sorted(self.images_dir.glob("*.png"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.labels_dir / f"{img_path.stem}.txt"
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        boxes, labels = [], []
        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    cid, xc, yc, w, h = line.split()
                    cid = int(cid)
                    xc, yc, w, h = (float(xc) * W, float(yc) * H, float(w) * W, float(h) * H)
                    boxes.append([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2])
                    labels.append(cid + 1)  # +1: class 0 is background in torchvision

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels_t = torch.as_tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)
        area = (boxes_t[:, 2] - boxes_t[:, 0]) * (boxes_t[:, 3] - boxes_t[:, 1]) if len(boxes_t) else torch.zeros((0,))

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(labels_t),), dtype=torch.int64),
        }
        img_tensor = torchvision.transforms.functional.to_tensor(img)
        return img_tensor, target

In [ ]:
def collate_fn(batch):
    # Detection batches have a variable number of boxes per image, so the
    # default collate (which tries to stack tensors) can't be used.
    return tuple(zip(*batch))

train_ds = CLCXrayDetectionDataset(DATASET_ROOT / "images/train", DATASET_ROOT / "labels/train")
val_ds = CLCXrayDetectionDataset(DATASET_ROOT / "images/val", DATASET_ROOT / "labels/val")
test_ds = CLCXrayDetectionDataset(DATASET_ROOT / "images/test", DATASET_ROOT / "labels/test")

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

## 4. Model — pretrained, fine-tuned on CLCXray

Using the built-in COCO-pretrained Faster R-CNN + MobileNetV3-Large + FPN (instructor-approved exception for this model — team's other two models still train from scratch, per the assignment's YOLO rule).

The pretrained head outputs COCO's classes, not ours, so — same idea as the assignment's classification transfer-learning rule ("remove the top layer, add new layers for your classes") — the box predictor gets swapped out for one sized to our 12 classes + background, while the pretrained backbone/FPN/RPN underneath stay in place as the starting point.

In [ ]:
from torchvision.models.detection import FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_mobilenet_v3_large_fpn(
    weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT,  # COCO-pretrained
)

# Replace the box predictor (classification + box-regression head) so it
# outputs our 13 classes instead of COCO's 91 -- the backbone/FPN/RPN keep
# their pretrained weights, only this head is newly initialized.
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model.to(DEVICE)
print("Model built. Backbone/FPN/RPN are COCO-pretrained; box predictor head is new.")

## 5. Training

**Gotcha**: `model.eval()` makes a torchvision detection model return *predictions*, not losses — there's no direct way to get a validation loss in eval mode. The standard workaround (used in `compute_val_loss` below) is to keep the model in `train()` mode, so it still computes losses, but wrap everything in `torch.no_grad()` so nothing is actually learned from the validation set.

In [ ]:
def train_one_epoch(model, optimizer, loader):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def compute_val_loss(model, loader):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        total_loss += sum(loss_dict.values()).item()
    return total_loss / len(loader)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses, val_losses = [], []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, optimizer, train_loader)
    val_loss = compute_val_loss(model, val_loader)
    lr_scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_fasterrcnn.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

torch.save(model.state_dict(), "/kaggle/working/last_fasterrcnn.pt")

## 6. Loss curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.savefig("/kaggle/working/loss_curve.png")
plt.show()

## 7. GFLOPs and parameter count

In [ ]:
from fvcore.nn import FlopCountAnalysis, parameter_count

model.eval()
dummy_input = [torch.rand(3, 640, 640).to(DEVICE)]
flops = FlopCountAnalysis(model, (dummy_input,))
print("GFLOPs:", flops.total() / 1e9)
print("Parameters:", parameter_count(model)[""])

## 8. mAP50 / mAP50-95 on the test set

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

model.load_state_dict(torch.load("/kaggle/working/best_fasterrcnn.pt"))
model.eval()

map_metric = MeanAveragePrecision(iou_type="bbox")
all_preds, all_targets = [], []
with torch.no_grad():
    for images, targets in test_loader:
        images_dev = [img.to(DEVICE) for img in images]
        preds = model(images_dev)
        all_preds.extend([{k: v.cpu() for k, v in p.items()} for p in preds])
        all_targets.extend(targets)

map_metric.update(all_preds, all_targets)
map_results = map_metric.compute()
print("mAP50:", map_results["map_50"].item())
print("mAP50-95:", map_results["map"].item())

## 9. Precision / Recall / F-score

`torchmetrics`' mAP integrates over the whole confidence curve, so it doesn't hand back a single precision/recall number the way Ultralytics does. This computes them directly via greedy IoU matching, at a fixed confidence (0.5) and IoU (0.5) threshold — **note this in the report**: it isn't computed identically to the YOLO models' Ultralytics-native precision/recall.

In [ ]:
def box_iou(box, boxes):
    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area_b = (box[2] - box[0]) * (box[3] - box[1])
    area_boxes = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area_b + area_boxes - inter + 1e-6)


CONF_THRESH = 0.5
IOU_THRESH = 0.5
tp, fp, fn = 0, 0, 0

for pred, target in zip(all_preds, all_targets):
    keep = pred["scores"] >= CONF_THRESH
    pred_boxes = pred["boxes"][keep]
    pred_labels = pred["labels"][keep]
    gt_boxes = target["boxes"]
    gt_labels = target["labels"]
    matched_gt = set()

    for pb, pl in zip(pred_boxes, pred_labels):
        if len(gt_boxes) == 0:
            fp += 1
            continue
        ious = box_iou(pb, gt_boxes)
        ious[gt_labels != pl] = 0  # only match same-class ground truth
        best_iou, best_idx = ious.max(0)
        if best_iou >= IOU_THRESH and best_idx.item() not in matched_gt:
            tp += 1
            matched_gt.add(best_idx.item())
        else:
            fp += 1
    fn += len(gt_boxes) - len(matched_gt)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}")

## 10. Confusion matrix

Object-detection confusion matrices aren't in `torchmetrics` — the `supervision` library is built for exactly this.

In [ ]:
import supervision as sv
import numpy as np

sv_predictions, sv_targets = [], []
for pred, target in zip(all_preds, all_targets):
    sv_predictions.append(sv.Detections(
        xyxy=pred["boxes"].numpy(),
        confidence=pred["scores"].numpy(),
        class_id=pred["labels"].numpy() - 1,  # shift back: drop the +1 background offset
    ))
    sv_targets.append(sv.Detections(
        xyxy=target["boxes"].numpy(),
        class_id=target["labels"].numpy() - 1,
    ))

confusion_matrix = sv.ConfusionMatrix.from_detections(
    predictions=sv_predictions,
    targets=sv_targets,
    classes=CLASS_NAMES,
)
confusion_matrix.plot().figure.savefig("/kaggle/working/confusion_matrix.png")

## 11. Sample prediction images for the report (6-7)

In [ ]:
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms.functional as TF

sample_paths = sorted((DATASET_ROOT / "images/test").glob("*.png"))[:7]
for i, img_path in enumerate(sample_paths):
    img = Image.open(img_path).convert("RGB")
    img_tensor = TF.to_tensor(img).to(DEVICE)
    with torch.no_grad():
        pred = model([img_tensor])[0]
    keep = pred["scores"] >= CONF_THRESH
    boxes = pred["boxes"][keep].cpu()
    labels = [CLASS_NAMES[l - 1] for l in pred["labels"][keep].cpu()]
    img_uint8 = (img_tensor.cpu() * 255).to(torch.uint8)
    drawn = draw_bounding_boxes(img_uint8, boxes, labels=labels, width=2)
    TF.to_pil_image(drawn).save(f"/kaggle/working/sample_pred_{i}.png")

print("Saved sample predictions to /kaggle/working/")

## 12. Package results for download

`/kaggle/working` is wiped once the session ends unless you commit the notebook or download files manually — do this before closing the tab.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/FasterRCNN_MobileNetV3_results", "zip", "/kaggle/working",
                     base_dir=".")
print("Zipped. Download FasterRCNN_MobileNetV3_results.zip from the file browser,")
print("or commit this notebook (Save Version -> Save & Run All) to preserve everything.")